# Notebook 01 - Data Extraction And Validation

This notebook prepares the United States real GDP series from 1920 onward and validates overlapping annual growth rates against FRED/BEA when the validation file is available.

## Research Question

When did United States real GDP grow above or below its own long-run average, and are the source data stable enough to support regime analysis?

## Data-Source Rationale

Maddison Project Database 2023 is the default historical source because it provides long-run GDP per capita and population estimates that cover 1920 onward. FRED/BEA `GDPCA` starts in 1929 and is therefore used as a validation source for modern national accounts rather than as the default 1920-onward series.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from us_gdp_regime.config import load_config
from us_gdp_regime.pipeline import prepare_data

config = load_config(Path("config/default.yaml"))
outputs = prepare_data(config)
outputs

## Maddison Extraction

The prepared series constructs total real GDP as real GDP per capita multiplied by population. Constant unit scaling does not affect annual growth rates or log-trend timing.

In [ ]:
series = pd.read_csv(outputs["series"])
series.head()

## Variable Definitions

In [ ]:
pd.DataFrame({
    "variable": ["year", "real_gdp", "gdp_growth", "real_gdp_per_capita", "population", "source"],
    "definition": [
        "Calendar year",
        "Maddison-derived real GDP proxy",
        "Annual percent change in real GDP",
        "Maddison real GDP per capita estimate",
        "Maddison population estimate",
        "Source identifier",
    ],
})

## Missing-Value Checks

In [ ]:
series.isna().sum().to_frame("missing_values")

## Growth-Rate Calculation Checks

In [ ]:
check = series[["year", "real_gdp", "gdp_growth"]].copy()
check["computed_growth"] = check["real_gdp"].pct_change() * 100
check["difference"] = check["gdp_growth"] - check["computed_growth"]
check.head(10)

## FRED/BEA Overlap Comparison

If a local FRED/BEA file is available, the pipeline writes the overlapping growth comparison and diagnostic summary. Missing FRED data does not block the Maddison-based analysis.

In [ ]:
comparison_path = config.paths.models_dir / "fred_maddison_growth_comparison.csv"
summary_path = config.paths.models_dir / "source_validation_summary.csv"
if comparison_path.exists():
    comparison = pd.read_csv(comparison_path)
    display(comparison.head())
    if summary_path.exists():
        display(pd.read_csv(summary_path))
else:
    comparison = pd.DataFrame()
    print("FRED/BEA validation file is not available; source validation was skipped.")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(series["year"], series["gdp_growth"], linewidth=1.2)
ax.set_title("United States real GDP annual growth")
ax.set_xlabel("Year")
ax.set_ylabel("Annual real GDP growth, %")
ax.grid(alpha=0.3)
ax.text(0.01, -0.18, "Source: Maddison Project Database 2023; GDP is per-capita GDP times population.", transform=ax.transAxes, fontsize=9)
fig.tight_layout()

## Export Of Cleaned Data

The cleaned data are written to `data/processed/us_gdp_series.csv`. If FRED/BEA validation is available, overlap diagnostics are written under `data/models/` and the comparison figure under `figures/`.

In [ ]:
outputs

## Short Conclusion

Maddison is the appropriate default for a 1920 start date because it covers the full historical window. FRED/BEA `GDPCA` is better treated as a validation source from 1929 onward. When the overlap diagnostics are available, use the correlation, absolute differences, and largest-difference years to judge whether the two growth series are close enough for regime analysis.